# Rithmic Adapter Sandbox Smoke

This notebook validates the Rithmic shadow-port branch in a dedicated sibling Nautilus clone. It is intentionally a smoke harness, not a production live-node workflow.

## Scope

The default path checks three things:

- the sandbox `.venv` can import the local Nautilus build
- low-level live quote subscription works against the current front month
- low-level historical minute-bar requests work through an explicit history-enabled gateway

Order-path cells are included, but they stay disabled unless you opt in with `RITHMIC_SANDBOX_RUN_*` environment flags.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import textwrap
from pathlib import Path


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from current working directory")


REPO_ROOT = find_repo_root()
PYTHON = REPO_ROOT / ".venv/bin/python"
PROFILE = os.environ.get("RITHMIC_PROFILE")
PRODUCT = os.environ.get("RITHMIC_SANDBOX_ROOT", "MNQ")
EXCHANGE = os.environ.get("RITHMIC_SANDBOX_EXCHANGE", "CME")
BAR_PERIOD = int(os.environ.get("RITHMIC_SANDBOX_BAR_PERIOD", "1"))
BAR_LOOKBACK_MINUTES = int(os.environ.get("RITHMIC_SANDBOX_BAR_LOOKBACK_MINUTES", "30"))
RUN_MARKET_DATA_SMOKE = os.environ.get("RITHMIC_SANDBOX_RUN_MARKET_DATA", "1") == "1"
RUN_BARS_SMOKE = os.environ.get("RITHMIC_SANDBOX_RUN_BARS", "1") == "1"
RUN_ORDER_SMOKE = os.environ.get("RITHMIC_SANDBOX_RUN_ORDER", "0") == "1"
RUN_BRACKET_SMOKE = os.environ.get("RITHMIC_SANDBOX_RUN_BRACKET", "0") == "1"
RUN_OCO_SMOKE = os.environ.get("RITHMIC_SANDBOX_RUN_OCO", "0") == "1"


def run_command(
    cmd: list[str], *, timeout: int = 120, extra_env: dict[str, str] | None = None
) -> subprocess.CompletedProcess[str]:
    printable = subprocess.list2cmdline(cmd)
    print(f"$ {printable}")
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)
    result = subprocess.run(  # noqa: S603
        cmd,
        cwd=REPO_ROOT,
        env=env,
        text=True,
        capture_output=True,
        timeout=timeout,
        check=False,
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr, file=sys.stderr)
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {printable}")
    if result.stderr:
        print(result.stderr)
    return result


def run_python(
    snippet: str, *, timeout: int = 120, extra_env: dict[str, str] | None = None
) -> subprocess.CompletedProcess[str]:
    return run_command(
        [str(PYTHON), "-c", textwrap.dedent(snippet)], timeout=timeout, extra_env=extra_env
    )

## Environment Summary

In [ ]:
summary = {
    "repo_root": str(REPO_ROOT),
    "python": str(PYTHON),
    "profile": PROFILE,
    "product": PRODUCT,
    "exchange": EXCHANGE,
    "bar_period": BAR_PERIOD,
    "bar_lookback_minutes": BAR_LOOKBACK_MINUTES,
    "run_market_data_smoke": RUN_MARKET_DATA_SMOKE,
    "run_bars_smoke": RUN_BARS_SMOKE,
    "run_order_smoke": RUN_ORDER_SMOKE,
    "run_bracket_smoke": RUN_BRACKET_SMOKE,
    "run_oco_smoke": RUN_OCO_SMOKE,
}
print(json.dumps(summary, indent=2))

## Import And Config Sanity

In [ ]:
run_python("""
import json
import os
import nautilus_trader
from nautilus_trader.adapters.rithmic import RithmicDataClientConfig
from nautilus_trader.adapters.rithmic import RithmicExecClientConfig

profile = os.environ.get('RITHMIC_PROFILE')
data = RithmicDataClientConfig.from_env(profile)
payload = {
    "nautilus_version": nautilus_trader.__version__,
    "data_environment": data.environment.value,
    "data_username": data.username,
    "data_system_name": data.system_name,
    "data_has_fcm_id": bool(data.fcm_id),
    "data_has_ib_id": bool(data.ib_id),
}
try:
    exec_config = RithmicExecClientConfig.from_env(profile)
except ValueError as exc:
    payload["exec_config"] = f"not available: {exc}"
else:
    payload["exec_environment"] = exec_config.environment.value
    payload["exec_account_id"] = exec_config.account_id
print(json.dumps(payload, indent=2))
""")

## Market Data Smoke

This uses the low-level PyO3 bindings provider for front-month resolution, matching the current `examples/live/rithmic/order_submission.py` flow. The high-level Python `RithmicInstrumentProvider` wrapper expects a Nautilus config object instead of a raw gateway, so it is not the right surface for this low-level smoke path.

In [ ]:
if RUN_MARKET_DATA_SMOKE:
    run_python(
        f"""
import asyncio
import json
import os
from nautilus_trader.adapters.rithmic import RithmicDataClient
from nautilus_trader.adapters.rithmic import RithmicGateway
from nautilus_trader.adapters.rithmic.bindings import RithmicInstrumentProvider

PRODUCT = {PRODUCT!r}
EXCHANGE = {EXCHANGE!r}
EVENT_TIMEOUT_SECONDS = 30.0

async def main() -> None:
    profile = os.environ.get('RITHMIC_PROFILE')
    gateway = RithmicGateway.from_env(profile)
    queue: asyncio.Queue = asyncio.Queue()
    client = None
    try:
        await gateway.connect()
        provider = RithmicInstrumentProvider(gateway)
        instrument = await provider.load_front_month_async(PRODUCT, EXCHANGE)
        client = RithmicDataClient(gateway)
        loop = asyncio.get_running_loop()
        client.set_data_callback(lambda event: loop.call_soon_threadsafe(queue.put_nowait, event))
        await client.start_event_loop()
        await client.subscribe_quotes(instrument.symbol, instrument.exchange)
        while True:
            event = await asyncio.wait_for(queue.get(), timeout=EVENT_TIMEOUT_SECONDS)
            if event.is_error():
                raise RuntimeError(event.as_error())
            if event.is_quote():
                quote = event.as_quote()
                if quote.bid_price > 0.0 and quote.ask_price > 0.0:
                    print(json.dumps({{
                        "symbol": instrument.symbol,
                        "exchange": instrument.exchange,
                        "tick_size": instrument.tick_size,
                        "currency": instrument.currency,
                        "bid_price": quote.bid_price,
                        "ask_price": quote.ask_price,
                    }}, indent=2))
                    break
    finally:
        if client is not None:
            client.unsubscribe_all()
            client.clear_data_callback()
            client.stop_event_loop()
        await gateway.disconnect()

asyncio.run(main())
""",
        timeout=90,
    )
else:
    print("Skipped market-data smoke; set RITHMIC_SANDBOX_RUN_MARKET_DATA=1 to enable.")

## Historical Bars Smoke

This uses an explicit history-enabled gateway. The convenience `RithmicGateway.from_env(...)` path keeps the history plant disabled by default, which is why the notebook does not reuse it for `request_bars(...)`.

In [ ]:
if RUN_BARS_SMOKE:
    run_python(
        f"""
import asyncio
import json
import os
import time
from nautilus_trader.adapters.rithmic import RithmicDataClient
from nautilus_trader.adapters.rithmic import RithmicDataClientConfig
from nautilus_trader.adapters.rithmic import RithmicGateway
from nautilus_trader.adapters.rithmic.bindings import RithmicInstrumentProvider
from nautilus_trader.core import nautilus_pyo3

PRODUCT = {PRODUCT!r}
EXCHANGE = {EXCHANGE!r}
BAR_PERIOD = {BAR_PERIOD}
LOOKBACK_MINUTES = {BAR_LOOKBACK_MINUTES}

def is_valid_bar(bar) -> bool:
    return bool(getattr(bar, "period", "")) and any(
        float(getattr(bar, attr, 0.0)) != 0.0
        for attr in ("open_price", "high_price", "low_price", "close_price", "volume")
    )

async def main() -> None:
    profile = os.environ.get('RITHMIC_PROFILE')
    base = RithmicDataClientConfig.from_env(profile)
    env = getattr(nautilus_pyo3.rithmic.RithmicEnv, base.environment.name)
    gateway = RithmicGateway(
        environment=env,
        username=base.username,
        password=base.password,
        system_name=base.system_name,
        app_name=base.app_name,
        app_version=base.app_version,
        fcm_id=base.fcm_id or "",
        ib_id=base.ib_id or "",
        account_id="",
        server=base.server,
        alt_server=base.alt_server,
        enable_ticker=True,
        enable_order=False,
        enable_pnl=False,
        enable_history=True,
    )
    try:
        await gateway.connect()
        provider = RithmicInstrumentProvider(gateway)
        instrument = await provider.load_front_month_async(PRODUCT, EXCHANGE)
        client = RithmicDataClient(gateway)
        end_sec = int(time.time())
        start_sec = end_sec - LOOKBACK_MINUTES * 60
        bars = await client.request_bars(
            instrument.symbol,
            instrument.exchange,
            "MinuteBar",
            BAR_PERIOD,
            start_sec,
            end_sec,
        )
        valid_bars = [bar for bar in bars if is_valid_bar(bar)]
        print(json.dumps({{
            "symbol": instrument.symbol,
            "exchange": instrument.exchange,
            "raw_bar_count": len(bars),
            "valid_bar_count": len(valid_bars),
            "first_valid_bar": repr(valid_bars[0]) if valid_bars else None,
            "last_valid_bar": repr(valid_bars[-1]) if valid_bars else None,
        }}, indent=2))
    finally:
        await gateway.disconnect()

asyncio.run(main())
""",
        timeout=90,
    )
else:
    print("Skipped bars smoke; set RITHMIC_SANDBOX_RUN_BARS=1 to enable.")

## Optional Order-Path Smokes

These cells call the existing example scripts from the sandbox repo. They are off by default because they can submit real orders to the configured account. Use demo credentials first.

In [ ]:
if RUN_ORDER_SMOKE:
    run_command([str(PYTHON), "examples/live/rithmic/order_submission.py"], timeout=180)
else:
    print("Skipped order smoke; set RITHMIC_SANDBOX_RUN_ORDER=1 to enable.")

In [ ]:
if RUN_BRACKET_SMOKE:
    run_command([str(PYTHON), "examples/live/rithmic/bracket_submission.py"], timeout=180)
else:
    print("Skipped bracket smoke; set RITHMIC_SANDBOX_RUN_BRACKET=1 to enable.")

In [ ]:
if RUN_OCO_SMOKE:
    run_command([str(PYTHON), "examples/live/rithmic/oco_submission.py"], timeout=180)
else:
    print("Skipped OCO smoke; set RITHMIC_SANDBOX_RUN_OCO=1 to enable.")

## Result

If the import, quote, and bars cells all pass, the sandbox clone is good enough for controlled adapter smoke work. Use the order-path cells only when you want an intentional demo-account routing check.